# Phase 4: Training Pipeline (cnn1d)

In [1]:
# Cell 1: Environment Setup & Hardware Detection
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_project_root(indicator_file="configs/datasets.yaml"):
    # Dynamically find the project root relative to the notebook location.
    curr = os.getcwd()
    # Check current and up to 4 parent levels (Notebooks are usually in notebooks/)
    for _ in range(5):
        if os.path.exists(os.path.join(curr, indicator_file)):
            return curr
        curr = os.path.dirname(curr)
    # Fallback for local dev or standard Colab location
    return '/content/drive/MyDrive/seismic-first-break-picking'

PROJECT_ROOT = find_project_root()

if IN_COLAB:
    # Ensure all required packages are installed in the fresh Colab runtime
    !pip install -q mlflow optuna lightgbm segmentation-models-pytorch pyyaml

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

import torch
import numpy as np
import random
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_type)
device_name = torch.cuda.get_device_name(0) if device_type == 'cuda' else 'cpu'
vram_bytes = torch.cuda.get_device_properties(0).total_memory if device_type == 'cuda' else 0

print(f"Device: {device_name}")
print(f"VRAM: {vram_bytes / 1e9:.2f} GB" if vram_bytes > 0 else "")


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [2]:
# Cell 2: Config & MLflow Setup
import mlflow
import types
from src.utils.config_loader import load_model_config
from src.training.mlflow_logger import MLFlowLogger

config_path = os.path.join(PROJECT_ROOT, 'configs', 'model_cnn1d.yaml')
config = load_model_config(config_path, as_namespace=True)

# Defensive fallback: guarantee config.output exists with all required keys
if not hasattr(config, 'output'):
    config.output = types.SimpleNamespace()
if not hasattr(config.output, 'tracking_uri'):
    config.output.tracking_uri = f'file:///{PROJECT_ROOT}/mlruns'
if not hasattr(config.output, 'checkpoint_dir'):
    config.output.checkpoint_dir = f'models/cnn1d'

# CRITICAL: Resolve checkpoint_dir to an absolute path inside Google Drive.
# If it is a relative path (e.g. 'models/cnn1d'), Colab resolves it against
# CWD=/content (ephemeral disk) � models would be LOST when the session ends.
# Prepend PROJECT_ROOT so all checkpoints land on Drive unconditionally.
if not os.path.isabs(config.output.checkpoint_dir):
    config.output.checkpoint_dir = os.path.join(PROJECT_ROOT, config.output.checkpoint_dir)
os.makedirs(config.output.checkpoint_dir, exist_ok=True)
print(f"Checkpoint dir (absolute): {config.output.checkpoint_dir}")

# Terminate any active MLflow run from a previous cell execution (rerun safety)
mlflow.end_run()

# --- Resume-safe MLflow init ---
# _save_checkpoint writes mlflow_run_id.txt on every epoch save.
# If the file exists, we rejoin that run so all epochs appear on one timeline.
# If the run was FINISHED (or the file is stale) resume_run() transparently
# falls back to a fresh run, so re-running a completed notebook is fully safe.
_run_id_file = os.path.join(config.output.checkpoint_dir, 'mlflow_run_id.txt')
_existing_run_id = None
if os.path.exists(_run_id_file):
    with open(_run_id_file) as _f:
        _existing_run_id = _f.read().strip() or None

logger = MLFlowLogger(config.output.tracking_uri, config.experiment.name)
if _existing_run_id:
    logger.resume_run(_existing_run_id)
    print(f"Resumed MLflow run: {_existing_run_id}")
else:
    logger.start_run()

logger.log_params(config)
print(f"Loaded config for: {config.experiment.name}")
print(f"Checkpoint dir: {config.output.checkpoint_dir}")


Checkpoint dir (absolute): /content/drive/MyDrive/seismic-first-break-picking/models/cnn1d


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/12 10:23:14 INFO mlflow.tracking.fluent: Experiment with name 'cnn1d_standard' does not exist. Creating a new experiment.


Resumed MLflow run: 6807972aefdb4b6da8861299627cf7b7
Loaded config for: cnn1d_standard
Checkpoint dir: /content/drive/MyDrive/seismic-first-break-picking/models/cnn1d


In [3]:
# Cell 3: Checkpoint State Detection
import os
import torch

# Trainer saves as '{experiment_name}_latest.pt' � match that filename exactly.
checkpoint_path = os.path.join(config.output.checkpoint_dir, f'{config.experiment.name}_latest.pt')
is_progressive = hasattr(config, 'progressive_training') and getattr(config.training, 'training_mode', 'combined') == 'progressive'

if not os.path.exists(checkpoint_path):
    state = 'NO_CHECKPOINT_EXISTS'
    start_asset_index = 0
    resume_epoch = 0
    completed_assets = []
else:
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    training_state = checkpoint.get('training_state', {})
    resume_epoch = checkpoint.get('epoch', 0)

    if is_progressive:
        if training_state.get('is_fully_trained', False):
            state = 'TRAINING_COMPLETE'
            completed_assets = config.progressive_training.asset_order
            start_asset_index = len(completed_assets)
        else:
            completed_assets = training_state.get('completed_assets', [])
            start_asset_index = len(completed_assets)
            state = f"ASSET_{start_asset_index}_COMPLETE" if completed_assets else "NO_CHECKPOINT_EXISTS"
    else:
        # Combined Training (Default)
        completed_assets = []
        start_asset_index = 0
        state = 'RESUMING_COMBINED_TRAINING'

print(f"Detected state: {state}")
print(f"Resuming from epoch: {resume_epoch}")


Detected state: RESUMING_COMBINED_TRAINING
Resuming from epoch: 20


In [4]:
# Cell 4: Data & Sampler Setup
from torch.utils.data import DataLoader
from src.data.dataset import ShotGatherDataset, ProgressiveAssetSampler, variable_width_collate_fn, trace_collate_fn
from src.data.transforms import build_transforms
from src.utils.config_loader import load_yaml

csv_path = config.data.index_csv if os.path.isabs(config.data.index_csv) else os.path.join(PROJECT_ROOT, config.data.index_csv)

# Load augmentation config and build training transforms
preproc_path = os.path.join(PROJECT_ROOT, 'configs', 'preprocessing.yaml')
preproc_cfg = load_yaml(preproc_path)
train_transform = build_transforms(preproc_cfg.get('augmentation', {}), is_training=True)
print(f"Train augmentations: {train_transform}")

train_ds = ShotGatherDataset(csv_path, split='train', transform=train_transform)
val_ds = ShotGatherDataset(csv_path, split='val')  # No augmentation for validation

val_loader = DataLoader(
    val_ds, batch_size=config.training.batch_size, shuffle=False,
    collate_fn=trace_collate_fn, num_workers=2, pin_memory=True
)

train_loader = DataLoader(
    train_ds, batch_size=config.training.batch_size, shuffle=True,
    collate_fn=trace_collate_fn, num_workers=2, pin_memory=True
)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")


Train augmentations: Compose([
])
Train samples: 2907 | Val samples: 621


In [5]:
# Cell 5: Model Map
from src.models.cnn_1d import Conv1DRegressor
model = Conv1DRegressor().to(device)
print(f"Parameters: {model.count_parameters()}")

Parameters: 184193


In [6]:
# Cell 6: Phase 4.7 Optuna Sweeper (Optional: Run ONLY on final best architecture)
try:
    import optuna
    from optuna.pruners import MedianPruner
except ImportError:
    optuna = None
    print("Install optuna for sweeps.")

def run_optuna_sweep(n_trials=30):
    if optuna is None: return

    print("WARNING: Do not run Optuna sweeps on all models. Compute limits apply.")
    print("Execute this function manually only on the singular best architecture.")

    def objective(trial):
        lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
        bs = trial.suggest_categorical("batch_size", [4, 8, 16] if '2d' in model_key else [32, 64, 128])
        # Insert discrete model instantiations + train epochs.
        # Report intermediate validation MAE to the pruner at each epoch:
        # trial.report(val_mae, epoch)
        # if trial.should_prune():
        #     raise optuna.TrialPruned()
        return 0.0 # Return absolute Val MAE here.

    study = optuna.create_study(direction="minimize", pruner=MedianPruner(n_warmup_steps=10))
    study.optimize(objective, n_trials=n_trials)
    print("Best params:", study.best_params)


In [7]:
# Cell 7: Execute Progressive Loop
from src.models.loss import MaskedMAELoss
from src.training.trainer import Trainer

criterion = MaskedMAELoss()

if state == 'TRAINING_COMPLETE':
    print("Skipping to evaluation")
else:
    # We iterate over the sequence of assets dynamically
    # For Combined mode, asset_order just has 'all'
    is_progressive = hasattr(config, 'progressive_training') and getattr(config.training, 'training_mode', 'combined') == 'progressive'

    if not is_progressive:
        optimizer = torch.optim.AdamW(model.parameters(), lr=config.training.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.training.epochs)
        trainer = Trainer(model, optimizer, criterion, config, device, scheduler, logger)
        # Resume from Drive checkpoint if one was detected in Cell 3
        if state == 'RESUMING_COMBINED_TRAINING' and os.path.isfile(checkpoint_path):
            trainer.load_checkpoint(checkpoint_path)
            print(f"Resumed from checkpoint at epoch {trainer.start_epoch - 1}")
        trainer.run(train_loader, val_loader)
    else:
        # Loop Progressive Assets
        for i in range(start_asset_index, len(config.progressive_training.asset_order)):
            current_asset = config.progressive_training.asset_order[i]
            print(f"--- Training Phase: {current_asset} ---")

            p_sampler = ProgressiveAssetSampler(
                train_ds,
                current_asset=current_asset,
                previous_assets=config.progressive_training.asset_order[:i],
                replay_fraction=config.progressive_training.replay_fraction
            )

            p_loader = DataLoader(
                train_ds, batch_size=config.training.batch_size, sampler=p_sampler,
                collate_fn=trace_collate_fn, num_workers=2, pin_memory=True
            )

            optimizer = torch.optim.AdamW(model.parameters(), lr=config.progressive_training.asset_lr[i])
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.progressive_training.asset_epochs[i])

            trainer = Trainer(model, optimizer, criterion, config, device, scheduler, logger)

            # On the first resumable asset, reload model weights from the checkpoint.
            # (The optimizer is intentionally NOT restored � each asset starts a fresh
            # optimiser phase with its own LR.  Only the learned weights carry over.)
            if i == start_asset_index and resume_epoch > 0 and os.path.isfile(checkpoint_path):
                _ckpt = torch.load(checkpoint_path, map_location=device)
                model.load_state_dict(_ckpt['model_state_dict'])
                print(f"  Loaded model weights from checkpoint (epoch {resume_epoch})")

            trainer.run(p_loader, val_loader,
                        start_epoch=resume_epoch,
                        total_epochs=config.progressive_training.asset_epochs[i])

            # Mark this asset complete inside the trainer so _save_checkpoint
            # persists the completed list to disk � prevents progressive amnesia
            # if the session dies before all assets finish.
            trainer.training_state = {
                'completed_assets': config.progressive_training.asset_order[:i + 1],
                'is_fully_trained': (i + 1) == len(config.progressive_training.asset_order),
            }
            trainer._save_checkpoint(
                config.progressive_training.asset_epochs[i], False,
                f"{config.experiment.name}_latest.pt"
            )

            # Reset resume so next asset starts at epoch 1
            resume_epoch = 0


Loading checkpoint from /content/drive/MyDrive/seismic-first-break-picking/models/cnn1d/cnn1d_standard_latest.pt
Resumed from epoch 20 with Best MAE: 151.5475
  Restored 20 history entries.
Resumed from checkpoint at epoch 20

--- Epoch 21/30 ---


Training:  12%|█▏        | 180/1454 [03:05<19:45,  1.07it/s, loss=88.7]/content/drive/MyDrive/seismic-first-break-picking/src/models/loss.py:31: RuntimeWarning: MaskedMAELoss received a batch with NO valid labels. Returning 0.0 loss with gradient attached.
  warnings.warn("MaskedMAELoss received a batch with NO valid labels. Returning 0.0 loss with gradient attached.", RuntimeWarning)
Training: 100%|██████████| 1454/1454 [24:19<00:00,  1.00s/it, loss=85.5]


Train | Loss: 138.8294 | MAE: 148.29ms
Val   | Loss: 153.5194 | MAE: 194.85ms | In-5ms: 2.11%

--- Epoch 22/30 ---


Training: 100%|██████████| 1454/1454 [19:31<00:00,  1.24it/s, loss=0]

Train | Loss: 139.1071 | MAE: 148.33ms


...

Train | Loss: 137.1108 | MAE: 147.68ms


Val   | Loss: 142.8182 | MAE: 168.37ms | In-5ms: 2.32%

--- Epoch 30/30 ---


Training: 100%|██████████| 1454/1454 [18:36<00:00,  1.30it/s, loss=75.1]

Train | Loss: 137.9487 | MAE: 147.56ms


Val   | Loss: 166.5042 | MAE: 215.04ms | In-5ms: 1.83%


In [8]:
# Cell 8: Universal Evaluation (all model tiers)
# Force-purge ALL src.* modules from Colab's module cache so that any fixes
# pushed to Drive (loss.py, trainer.py, dataset.py, evaluator.py, etc.) are
# reloaded fresh. Without this, Colab reuses stale bytecode from the session's
# first import, silently ignoring edits on Drive.
import importlib, sys
_stale = [k for k in sys.modules if k.startswith('src.')]
for _mod_name in _stale:
    del sys.modules[_mod_name]
print(f"Cache cleared: {len(_stale)} src.* modules evicted.")

from src.training.evaluator import ModelEvaluator

_is_dl = 'cnn1d' not in ('classical', 'tabular_lgbm')
_history = trainer.history if 'trainer' in dir() and hasattr(trainer, 'history') else {}

evaluator = ModelEvaluator(
    model=model,
    val_loader=val_loader,
    logger=logger,
    device=device,
    model_key='cnn1d',
    is_dl=_is_dl,
    history=_history,
)
final_metrics = evaluator.run()

# Close MLflow run cleanly
import mlflow
mlflow.end_run()
print("MLflow run closed.")


Cache cleared: 14 src.* modules evicted.

MODEL EVALUATOR: cnn1d

──────────────────────────────────────────────────
  VALIDATION METRICS
──────────────────────────────────────────────────
  MAE (mean):      215.11 ms
  MAE P50:         161.68 ms
  MAE P90:         491.33 ms
  MAE P95:         609.05 ms
  Within   5ms:    1.8%
  Within  10ms:    3.6%
  Within  20ms:    7.2%
  Parameters:      184,193
  Latency/trace:   0.049 ms
  Throughput:      20322 traces/s

  PER-ASSET BREAKDOWN
  brunswick    | MAE:  265.6 ms | <=5ms:   1.3%  (553838 traces)
  halfmile     | MAE:  197.7 ms | <=5ms:   1.5%  (145461 traces)
  lalor        | MAE:  100.6 ms | <=5ms:   3.2%  (184362 traces)
  sudbury      | MAE:   68.3 ms | <=5ms:   5.0%  (29335 traces)
──────────────────────────────────────────────────


/content/drive/MyDrive/seismic-first-break-picking/src/training/evaluator.py:288: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.tight_layout()



All metrics and plots logged to MLflow.
MLflow run closed.
